# Projet 1 - Exploration et Restitution de Données
## Sujet 7 : Mix Électrique Régional (RTE / éCO2mix)

### Contexte et Objectifs
Ce notebook répond aux exigences d'analyse exploratoire (**EF-01 à EF-05**) du cahier des charges.
L'objectif est de comprendre le profil énergétique des régions françaises à partir du jeu de données officiel **éCO2mix régional** (RTE/ODRE) :
1. Évaluer la part réelle des énergies renouvelables dans la consommation régionale.
2. Mesurer la dépendance inter-régionale (échanges physiques et pics hivernaux).
3. Analyser la saisonnalité et la dispersion des filières.
4. Tester les hypothèses métier formulées.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

# Configuration du style visuel conforme aux normes d'accessibilité
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

DATA_PATH = Path("../data/processed/eco2mix-regional-cons-def_clean.csv")
print("Bibliothèques chargées avec succès.")

---
## 1. Chargement et Profilage du Jeu de Données (Exigence EF-02)
Vérification des dimensions, types, cardinalités et couverture temporelle.

In [ ]:
# Chargement des données nettoyées avec typage adapté
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    parse_dates=["Date", "Date - Heure"],
    low_memory=False,
)

print(f"Dimensions du jeu de données : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Période couverte : du {df['Date'].min().strftime('%d/%m/%Y')} au {df['Date'].max().strftime('%d/%m/%Y')}")
print(f"Nombre de régions administratives : {df['Région'].nunique()}")
df.head(3)

In [ ]:
# Profil synthétique des données et valeurs manquantes
missing_summary = pd.DataFrame({
    "Type": df.dtypes,
    "Valeurs Manquantes": df.isnull().sum(),
    "Part Manquante (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary[missing_summary["Valeurs Manquantes"] > 0].sort_values("Part Manquante (%)", ascending=False).head(10)

---
## 2. Distributions et Indicateurs Clés (Exigence EF-03)
### Justification Moyenne vs Médiane
* **Moyenne :** Indispensable pour les bilans énergétiques globaux (puissance moyenne en MW multipliée par le temps pour obtenir des MWh).
* **Médiane :** Préférée pour analyser le niveau de consommation typique et la disponibilité des EnR, car elle est robuste aux valeurs extrêmes (vagues de grand froid, tempêtes éoliennes).

In [ ]:
key_cols = ["Consommation (MW)", "Nucléaire (MW)", "Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)", "Thermique (MW)"]
stats_desc = df[key_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]
stats_desc.rename(columns={"50%": "médiane"}, inplace=True)
stats_desc

In [ ]:
# Distribution de la consommation régionale
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Consommation (MW)", y="Région", orient="h", palette="viridis")
plt.title("Distribution de la consommation électrique par région (MW) - RTE")
plt.xlabel("Puissance appelée (MW)")
plt.ylabel("Région")
plt.xlim(left=0)
plt.show()

---
## 3. Agrégations et Analyses Croisées (Exigence EF-04)
Agrégation temporelle (évolution mensuelle/annuelle) et géographique par filière.

In [ ]:
# Calcul de la production renouvelable totale (EnR)
enr_cols = ["Eolien (MW)", "Solaire (MW)", "Hydraulique (MW)", "Bioénergies (MW)"]
df["Production_EnR (MW)"] = df[enr_cols].sum(axis=1)

# Agrégation mensuelle nationale
df["Annee_Mois"] = df["Date"].dt.to_period("M")
monthly_agg = df.groupby("Annee_Mois")[["Consommation (MW)", "Production_EnR (MW)", "Nucléaire (MW)"]].mean()

plt.figure(figsize=(14, 6))
plt.plot(monthly_agg.index.astype(str), monthly_agg["Consommation (MW)"], label="Consommation moyenne (MW)", color="#1f77b4", lw=2)
plt.plot(monthly_agg.index.astype(str), monthly_agg["Production_EnR (MW)"], label="Production EnR (MW)", color="#2ca02c", lw=2)
plt.plot(monthly_agg.index.astype(str), monthly_agg["Nucléaire (MW)"], label="Nucléaire (MW)", color="#ff7f0e", lw=2, linestyle="--")
plt.title("Évolution mensuelle de la consommation et production électrique moyenne (MW)")
plt.xlabel("Mois")
plt.ylabel("Puissance moyenne (MW)")
plt.xticks(rotation=45)
plt.legend()
plt.ylim(bottom=0)
plt.show()

---
## 4. Formulation et Vérification d'Hypothèses (Exigence EF-05)

### Hypothèse 1 : Saisonnalité et déficit de couverture des EnR en hiver
* **Formulation :** *En période hivernale (décembre à février), le pic de consommation coïncide avec une faible production solaire, entraînant une dépendance accrue au pilotable ou aux importations.*
* **Test :** Calcul du taux de couverture EnR moyen (Production EnR / Consommation) par saison.

In [ ]:
df["Mois"] = df["Date"].dt.month
df["Saison"] = df["Mois"].map({
    12: "Hiver", 1: "Hiver", 2: "Hiver",
    3: "Printemps", 4: "Printemps", 5: "Printemps",
    6: "Été", 7: "Été", 8: "Été",
    9: "Automne", 10: "Automne", 11: "Automne"
})

saison_stats = df.groupby("Saison").apply(
    lambda g: pd.Series({
        "Consommation_Moy (MW)": g["Consommation (MW)"].mean(),
        "EnR_Moy (MW)": g["Production_EnR (MW)"].mean(),
        "Taux_Couverture_EnR (%)": (g["Production_EnR (MW)"].sum() / g["Consommation (MW)"].sum() * 100)
    })
)
saison_stats

### Hypothèse 2 : Dépendance structurelle aux échanges inter-régionaux
* **Formulation :** *Certaines régions métropolitaines (notamment l'Île-de-France) sont en déficit chronique de production locale et présentent un solde d'échanges physiques positif (importateur net) sur l'ensemble de l'année.*
* **Test :** Somme et moyenne des échanges physiques par région.

In [ ]:
echanges_region = df.groupby("Région")["Ech. physiques (MW)"].agg(["mean", "sum"]).sort_values("mean", ascending=False)
echanges_region.rename(columns={
    "mean": "Solde moyen (MW)",
    "sum": "Solde cumulé"
}, inplace=True)
echanges_region["Profil"] = np.where(echanges_region["Solde moyen (MW)"] > 0, "Importateur net", "Exportateur net")
echanges_region